# `argmin` vs `argmax` dans `get_role_from_skills`

Le bug corrigé par le commit `35a8486`, sur les vraies données du projet.

`get_role_from_skills` construit une matrice booléenne `conditions_met[niveau, pompier]`,
vraie quand le pompier satisfait toutes les contraintes de compétence du niveau de rôle.
On veut **le premier niveau compatible** de chaque pompier.

| | renvoie | signification |
|---|---|---|
| `np.argmax` | premier `True` | premier rôle **compatible** ✅ |
| `np.argmin` | premier `False` | premier rôle **incompatible** ❌ |

Les deux copies de la thèse (`TheSachari/Thesis` et `Git_version_final`) portent `argmin`
à la ligne 1041 de leur `collective_functions.py` ; le code actuel porte `argmax` à la
ligne 1348.

In [ ]:
import os
from pathlib import Path

REPO = Path.cwd()
os.environ.setdefault("RL_DATA_ROOT", str(REPO / "run_full"))

import numpy as np
import pandas as pd

from paths import DATA_ENVIRONMENT
from collective_functions import (
    generate_dic_roles_skills,
    ff_match_operands,
    _role_operands,
    update_skills,
)

print("data root :", os.environ["RL_DATA_ROOT"])

## 1. Charger le minimum : compétences des pompiers et définitions de rôles

`df_skills` porte, par pompier et par compétence, une fenêtre de validité
(`Début`/`Fin`). `update_skills` la réduit à une matrice binaire à une date donnée :
`ff_array[pompier, compétence] ∈ {0, 1}`.

`dic_roles_skills[fonction]` est une matrice `(niveaux, compétences)` à valeurs dans
`{-1, 0, 1}` : `1` = compétence exigée, `-1` = compétence interdite, `0` = indifférent.

In [ ]:
df_skills = pd.read_pickle(DATA_ENVIRONMENT / "df_skills.pkl")
df_roles = pd.read_pickle(DATA_ENVIRONMENT / "df_roles.pkl")

dic_roles_skills = generate_dic_roles_skills(df_roles, df_skills)

skill_names = list(df_skills.columns.get_level_values(0).unique())

ff_array = update_skills(df_skills, pd.Timestamp("2018-06-15 14:00:00"))

print(f"pompiers          : {ff_array.shape[0]}")
print(f"compétences       : {ff_array.shape[1]}")
print(f"fonctions (rôles) : {len(dic_roles_skills)}")

## 2. Les deux versions, côte à côte

Corps de `get_role_from_skills`, au caractère près, la réduction laissée en paramètre.
Le `+1` décale les rôles à partir de 1 ; la dernière ligne remet à `0` les pompiers
compatibles avec *aucun* niveau (ce filet fonctionnait dans les deux versions, ce qui
contribuait à masquer le bug).

In [ ]:
def conditions_matrix(required_skills, ff_array):
    """conditions_met[niveau, pompier] : le pompier satisfait-il ce niveau ?"""
    plus, minus, base = _role_operands(required_skills)
    f_zero_t, f_one_t = ff_match_operands(ff_array)
    violations = base - plus @ f_one_t - minus @ f_zero_t
    return violations == 0


def role_from_skills(required_skills, ff_array, reduce):
    conditions_met = conditions_matrix(required_skills, ff_array)
    first_valid_index = reduce(conditions_met, axis=0) + 1
    first_valid_index[~np.any(conditions_met, axis=0)] = 0
    return first_valid_index


role_argmin = lambda rs, ff: role_from_skills(rs, ff, np.argmin)  # thèse
role_argmax = lambda rs, ff: role_from_skills(rs, ff, np.argmax)  # actuel

## 3. Un exemple minimal, entièrement lisible

Trois niveaux de rôle, trois pompiers construits à la main pour montrer les trois cas.

In [ ]:
# 4 compétences fictives ; 3 niveaux de rôle
toy_roles = np.array([
    [1, 1, 0, 0],   # niveau 1 : exige A et B
    [1, 0, 0, 0],   # niveau 2 : exige A
    [0, 0, 1, 0],   # niveau 3 : exige C
], dtype=float)

toy_ff = np.array([
    [1, 1, 0, 0],   # pompier 0 : A+B  -> compatible niveaux 1 et 2
    [0, 0, 1, 0],   # pompier 1 : C    -> compatible niveau 3 seulement
    [0, 0, 0, 1],   # pompier 2 : D    -> compatible avec aucun
], dtype=float)

cm = conditions_matrix(toy_roles, toy_ff)

print("conditions_met (lignes = niveaux, colonnes = pompiers)")
print(pd.DataFrame(cm, index=[f"niveau {i+1}" for i in range(3)],
                   columns=[f"pompier {j}" for j in range(3)]))

print("\nargmin (thèse) :", role_argmin(toy_roles, toy_ff))
print("argmax (actuel):", role_argmax(toy_roles, toy_ff))

Lecture attendue :

- **pompier 0** — `[True, True, False]`. `argmax` → niveau 1, qu'il peut tenir.
  `argmin` renvoie le premier `False`, donc le niveau 3, le seul qui lui soit interdit.
- **pompier 1** — `[False, False, True]`. `argmax` → niveau 3, correct.
  `argmin` → niveau 1, incompatible.
- **pompier 2** — compatible avec aucun niveau : les deux versions renvoient `0`,
  grâce à la ligne de garde. C'est le seul cas où elles s'accordent.

## 4. Sur les vraies données : quelques pompiers, quelques rôles

On prend une fonction réelle à plusieurs niveaux et on détaille des pompiers pour
lesquels les deux versions divergent.

## 3 bis. Le rôle `*CDG*` (chef de groupe), cas d'école

`*CDG*` est l'exemple le plus lisible du dépôt : 4 niveaux, **une seule compétence
exigée par niveau**, aucune compétence interdite.

| niveau | compétence exigée |
|---|---|
| 1 | `CDGROUPE_INC` (incendie) |
| 2 | `CDGROUPE_SAP` (secours à personne) |
| 3 | `CDGROUPE_DIV` (divers) |
| 4 | `CDCOLONNE` (chef de colonne) |

Comme les niveaux sont mutuellement exclusifs dans les faits — un chef de groupe
détient le triplet `INC`/`SAP`/`DIV`, ou bien `CDCOLONNE` — l'inversion `argmin`
envoie presque systématiquement chacun vers le rôle de l'autre.

In [ ]:
FONCTION = "*CDG*"
required = dic_roles_skills[FONCTION]

# --- ce que le rôle exige, niveau par niveau ---
print(f"=== {FONCTION} : {required.shape[0]} niveaux ===\n")
print(df_roles[df_roles["Fonction"] == FONCTION][["Ordre", "Competences"]]
      .to_string(index=False))

cdg_skills = []
print()
for lvl in range(required.shape[0]):
    exigees = [skill_names[s] for s in np.flatnonzero(required[lvl] == 1)]
    interdites = [skill_names[s] for s in np.flatnonzero(required[lvl] == -1)]
    cdg_skills += exigees
    print(f"  niveau {lvl+1} : exige {exigees}"
          + (f", interdit {interdites}" if interdites else ""))

# --- ce que les pompiers détiennent, sur ces mêmes compétences ---
cm = conditions_matrix(required, ff_array)
r_min = role_argmin(required, ff_array)
r_max = role_argmax(required, ff_array)

qualifies = np.flatnonzero(cm.any(axis=0))
divergents = np.flatnonzero((r_min != r_max) & (r_max > 0))
print(f"\npompiers compatibles avec >= 1 niveau : {qualifies.size} / {ff_array.shape[0]}")
print(f"dont réponse divergente argmin/argmax : {divergents.size}"
      f" ({100*divergents.size/max(qualifies.size,1):.0f} % des qualifiés)")

cols = [skill_names.index(c) for c in cdg_skills]
rows = []
for f in divergents[:10]:
    rows.append({
        "pompier": df_skills.index[f],
        **{c: int(ff_array[f, ci]) for c, ci in zip(cdg_skills, cols)},
        "compatible": list(np.flatnonzero(cm[:, f]) + 1),
        "argmin": r_min[f],
        "argmax": r_max[f],
        "argmin ok ?": "oui" if cm[r_min[f] - 1, f] else "NON",
    })
display(pd.DataFrame(rows).set_index("pompier"))

### 3 ter. Le détail de `conditions_met`, avant et après le `np.all`

La version thèse teste littéralement chaque triplet *(niveau, pompier, compétence)* :

```python
matches_minus_one   = (required_skills == -1)[:, np.newaxis, :]
matches_one         = (required_skills ==  1)[:, np.newaxis, :]
matches_zero_or_any = (required_skills ==  0)[:, np.newaxis, :]

conditions_met = ((matches_minus_one & (ff_array == 0)[np.newaxis, :, :]) |
                  (matches_one       & (ff_array == 1)[np.newaxis, :, :]) |
                  matches_zero_or_any)

conditions_met = np.all(conditions_met, axis=2)
```

Les trois masques couvrent les trois valeurs possibles d'une contrainte : `-1` satisfaite
si le pompier **n'a pas** la compétence, `1` s'il **l'a**, `0` toujours satisfaite. Leur
union dit, case par case, si *cette* contrainte tient.

Le `np.newaxis` aligne les axes pour le broadcast : le rôle devient
`(niveaux, 1, 134)` et les pompiers `(1, pompiers, 134)`, d'où un cube
**`(niveaux, pompiers, 134)`**. C'est ce cube que le refactor a éliminé — ~86 k booléens
par rôle, ~37 rôles par décision, le coût dominant de la simulation au profilage.

Le `np.all(axis=2)` réduit ensuite les 134 compétences en un seul verdict par
*(niveau, pompier)* : compatible seulement si **toutes** les contraintes tiennent.

In [ ]:
def conditions_met_thesis(required_skills, ff_array, reduce_all=True):
    """Le calcul de la thèse, tel quel. `reduce_all=False` rend le cube intermédiaire."""
    matches_minus_one   = (required_skills == -1)[:, np.newaxis, :]
    matches_one         = (required_skills ==  1)[:, np.newaxis, :]
    matches_zero_or_any = (required_skills ==  0)[:, np.newaxis, :]

    cube = ((matches_minus_one & (ff_array == 0)[np.newaxis, :, :]) |
            (matches_one       & (ff_array == 1)[np.newaxis, :, :]) |
            matches_zero_or_any)
    return np.all(cube, axis=2) if reduce_all else cube


# Deux pompiers représentatifs : un chef de groupe (INC/SAP/DIV) et un chef de colonne.
echantillon = [1029, 1033]
pos = [df_skills.index.get_loc(p) for p in echantillon]
sub = ff_array[pos]

cube = conditions_met_thesis(required, sub, reduce_all=False)

print("required_skills                :", required.shape)
print("  -> masques (newaxis)         :", (required == 1)[:, np.newaxis, :].shape)
print("ff_array (échantillon)         :", sub.shape)
print("  -> (newaxis)                 :", (sub == 1)[np.newaxis, :, :].shape)
print(f"\nconditions_met AVANT np.all    : {cube.shape}  = {cube.size} booléens")
print(f"  (sur les {ff_array.shape[0]} pompiers réels : "
      f"{required.shape[0] * ff_array.shape[0] * required.shape[1]:,} booléens par rôle)")

print("\ncontraintes satisfaites / 134, par (niveau, pompier) :")
display(pd.DataFrame(cube.sum(axis=2),
                     index=[f"niveau {i+1}" for i in range(cube.shape[0])],
                     columns=[f"ff {p}" for p in echantillon]))

cm2 = np.all(cube, axis=2)
print("conditions_met APRÈS np.all(axis=2) :", cm2.shape)
display(pd.DataFrame(cm2,
                     index=[f"niveau {i+1}" for i in range(cm2.shape[0])],
                     columns=[f"ff {p}" for p in echantillon]))
print("Une seule contrainte violée sur 134 suffit à rendre le niveau incompatible.")

In [ ]:
# Contribution de chaque masque, restreinte aux 4 compétences que *CDG* met en jeu.
cdg_cols = [skill_names.index(c) for c in cdg_skills]

m_minus = (required == -1)[:, np.newaxis, :]
m_one   = (required ==  1)[:, np.newaxis, :]
m_zero  = (required ==  0)[:, np.newaxis, :]

c_minus = m_minus & (sub == 0)[np.newaxis, :, :]
c_one   = m_one   & (sub == 1)[np.newaxis, :, :]
c_zero  = np.broadcast_to(m_zero, c_one.shape)

for f, ff_id in enumerate(echantillon):
    detenues = [c for c, ci in zip(cdg_skills, cdg_cols) if sub[f, ci] == 1]
    print(f"\n=== pompier {ff_id} — détient {detenues} ===")
    for lvl in range(required.shape[0]):
        tab = pd.DataFrame(
            {
                "contrainte du rôle": required[lvl, cdg_cols].astype(int),
                "détenue par le ff": sub[f, cdg_cols].astype(int),
                "masque == -1 & ff==0": c_minus[lvl, f, cdg_cols].astype(int),
                "masque ==  1 & ff==1": c_one[lvl, f, cdg_cols].astype(int),
                "masque ==  0 (indiff.)": c_zero[lvl, f, cdg_cols].astype(int),
                "union -> satisfaite ?": cube[lvl, f, cdg_cols].astype(int),
            },
            index=cdg_skills,
        )
        violations = int((~cube[lvl, f]).sum())
        verdict = "COMPATIBLE" if violations == 0 else f"INCOMPATIBLE ({violations} violation/134)"
        print(f"\n  niveau {lvl+1} — {verdict}")
        display(tab)

### 3 quater. Ce que fait chaque masque, isolément

`*CDG*` n'utilise que des `1` et des `0` : le masque `== -1` n'y sert jamais. Pour voir
les trois à l'œuvre, on prend **`COND_VSAV`** (conducteur d'ambulance), dont la grammaire
`A - B` se lit « exige A, **interdit** B » :

| niveau | expression | exige | interdit |
|---|---|---|---|
| 1 | `COND_AMBULANCE - CA_INC - CDGROUPE_SAP` | `COND_AMBULANCE` | `CA_INC`, `CDGROUPE_SAP` |
| 2 | `COND_AMBULANCE - CDGROUPE_SAP` | `COND_AMBULANCE` | `CDGROUPE_SAP` |
| 3 | `COND_VL_ROUTE` | `COND_VL_ROUTE` | — |

Le `-1` a un sens opérationnel : on ne met pas au volant quelqu'un dont les qualifications
sont nécessaires ailleurs. Un chef de groupe SAP est *écarté* du rôle de conducteur, pour
rester disponible au commandement.

Chacun des trois masques répond donc à une question différente, sur **la même case**
*(niveau, pompier, compétence)* :

| masque | contrainte visée | satisfait quand |
|---|---|---|
| `matches_one & (ff == 1)` | `+1` — exigée | le pompier **détient** la compétence |
| `matches_minus_one & (ff == 0)` | `-1` — interdite | le pompier **ne détient pas** la compétence |
| `matches_zero_or_any` | `0` — indifférente | **toujours** (aucune contrainte) |

Les trois sont mutuellement exclusifs — une contrainte vaut `-1`, `1` **ou** `0` — donc
leur `|` est en réalité une partition : pour chaque case, exactement un masque décide.
La ligne suivante le vérifie.

In [ ]:
FN2 = "COND_VSAV"
req2 = dic_roles_skills[FN2]

# Les compétences que ce rôle met en jeu (les 130 autres sont indifférentes partout).
en_jeu = sorted({s for lvl in req2 for s in np.flatnonzero(lvl != 0)})
noms2 = [skill_names[s] for s in en_jeu]

print(f"=== {FN2} — contraintes par niveau ===")
display(pd.DataFrame(req2[:, en_jeu].astype(int),
                     index=[f"niveau {i+1}" for i in range(req2.shape[0])],
                     columns=noms2))
print("  1 = exigée   |   -1 = interdite   |   0 = indifférente\n")

# Trois pompiers choisis pour activer chacun un masque différent au niveau 1.
i_amb, i_sap = skill_names.index("COND_AMBULANCE"), skill_names.index("CDGROUPE_SAP")
cand_ok  = np.flatnonzero((ff_array[:, i_amb] == 1) & (ff_array[:, i_sap] == 0))
cand_bad = np.flatnonzero((ff_array[:, i_amb] == 1) & (ff_array[:, i_sap] == 1))
cand_non = np.flatnonzero(ff_array[:, i_amb] == 0)
choisis = [c[0] for c in (cand_ok, cand_bad, cand_non) if c.size]
etiquettes = ["conducteur (amb, pas CDG_SAP)", "amb + CDG_SAP", "sans COND_AMBULANCE"]

sub2 = ff_array[choisis]
print("=== ce que détiennent ces pompiers ===")
display(pd.DataFrame(sub2[:, en_jeu].astype(int),
                     index=[f"{df_skills.index[c]} — {e}" for c, e in zip(choisis, etiquettes)],
                     columns=noms2))

# --- les trois masques, séparément, au niveau 1 ---
mm = (req2 == -1)[:, np.newaxis, :]
mo = (req2 ==  1)[:, np.newaxis, :]
mz = (req2 ==  0)[:, np.newaxis, :]

part_minus = mm & (sub2 == 0)[np.newaxis, :, :]
part_one   = mo & (sub2 == 1)[np.newaxis, :, :]
part_zero  = np.broadcast_to(mz, part_one.shape)
union      = part_minus | part_one | part_zero

LVL = 0
print(f"\n=== niveau {LVL+1} : qui satisfait quoi, masque par masque ===")
for f, (c, e) in enumerate(zip(choisis, etiquettes)):
    tab = pd.DataFrame(
        {
            "contrainte": req2[LVL, en_jeu].astype(int),
            "détenue": sub2[f, en_jeu].astype(int),
            "masque -1 (interdite)": part_minus[LVL, f, en_jeu].astype(int),
            "masque +1 (exigée)": part_one[LVL, f, en_jeu].astype(int),
            "masque 0 (indiff.)": part_zero[LVL, f, en_jeu].astype(int),
            "union": union[LVL, f, en_jeu].astype(int),
        },
        index=noms2,
    )
    viol = int((~union[LVL, f]).sum())
    print(f"\n  ff {df_skills.index[c]} — {e} : "
          + ("COMPATIBLE" if viol == 0 else f"INCOMPATIBLE ({viol} violation/134)"))
    display(tab)

# --- la partition ---
recouvre = (part_minus.astype(int) + part_one.astype(int) + part_zero.astype(int))
print("\nchaque case est décidée par au plus un masque :", bool((recouvre <= 1).all()))
print("cases satisfaites par le masque 0 (contrainte absente) :",
      f"{100*part_zero.mean():.1f} % — les 130 compétences hors rôle")

In [ ]:
def show_examples(fonction, n=6, seed=0):
    required = dic_roles_skills[fonction]
    cm = conditions_matrix(required, ff_array)
    r_min = role_argmin(required, ff_array)
    r_max = role_argmax(required, ff_array)

    differ = np.flatnonzero((r_min != r_max) & (r_max > 0))
    if differ.size == 0:
        print(f"{fonction} : aucune divergence")
        return

    picked = np.random.default_rng(seed).choice(
        differ, size=min(n, differ.size), replace=False
    )

    print(f"=== {fonction} — {required.shape[0]} niveaux de rôle ===")
    print(f"divergences : {differ.size} / {ff_array.shape[0]} pompiers "
          f"({100*differ.size/ff_array.shape[0]:.1f} %)\n")

    rows = []
    for f in picked:
        compatibles = np.flatnonzero(cm[:, f]) + 1
        held = [skill_names[s] for s in np.flatnonzero(ff_array[f])]
        rows.append({
            "pompier": df_skills.index[f],
            "compétences détenues": len(held),
            "niveaux compatibles": list(compatibles),
            "argmin (thèse)": r_min[f],
            "argmax (actuel)": r_max[f],
            "thèse valide ?": "oui" if r_min[f] in compatibles else "NON",
        })
    display(pd.DataFrame(rows).set_index("pompier"))


multi = [f for f, m in dic_roles_skills.items() if m.shape[0] > 2]
show_examples(multi[0])

In [ ]:
for fonction in multi[1:4]:
    show_examples(fonction, n=4)
    print()

## 5. Détail d'un cas : quelles compétences, quel rôle

Pour un pompier divergent, on affiche les contraintes du rôle que chaque version lui
attribue, et lesquelles il viole.

In [ ]:
def explain(fonction, ff_idx):
    required = dic_roles_skills[fonction]
    cm = conditions_matrix(required, ff_array)
    r_min = role_argmin(required, ff_array)[ff_idx]
    r_max = role_argmax(required, ff_array)[ff_idx]

    print(f"fonction {fonction} | pompier {df_skills.index[ff_idx]}")
    print(f"niveaux compatibles : {list(np.flatnonzero(cm[:, ff_idx]) + 1)}")
    print(f"argmin (thèse) -> niveau {r_min} | argmax (actuel) -> niveau {r_max}\n")

    for label, lvl in (("argmin (thèse)", r_min), ("argmax (actuel)", r_max)):
        if lvl == 0:
            print(f"{label}: aucun rôle")
            continue
        vec = required[lvl - 1]
        exigees = np.flatnonzero(vec == 1)
        interdites = np.flatnonzero(vec == -1)
        manquantes = [skill_names[s] for s in exigees if ff_array[ff_idx, s] != 1]
        en_trop = [skill_names[s] for s in interdites if ff_array[ff_idx, s] != 0]
        verdict = "COMPATIBLE" if not manquantes and not en_trop else "INCOMPATIBLE"
        print(f"{label} — niveau {lvl} : {verdict}")
        print(f"    exige {len(exigees)} compétence(s), interdit {len(interdites)}")
        if manquantes:
            print(f"    manquantes : {manquantes[:5]}")
        if en_trop:
            print(f"    interdites détenues : {en_trop[:5]}")
        print()


fonction = multi[0]
required = dic_roles_skills[fonction]
d = np.flatnonzero(
    (role_argmin(required, ff_array) != role_argmax(required, ff_array))
    & (role_argmax(required, ff_array) > 0)
)
explain(fonction, d[0])

## 6. Ampleur globale

Mesure sur les données effectivement chargées ici.

Le message du commit `35a8486` annonçait 31,8 % de recherches divergentes sur
111 / 117 fonctions multi-rôles ; il portait sur l'arbre de données de l'époque.
Sur le `Data_environment` courant on obtient plutôt ~15 % sur 73 / 116 fonctions —
la conclusion est la même, seule l'amplitude diffère.

In [ ]:
total = diverge = 0
invalides = 0
multi_role = 0

for fonction, required in dic_roles_skills.items():
    if required.shape[0] > 1:
        multi_role += 1
    cm = conditions_matrix(required, ff_array)
    r_min = role_argmin(required, ff_array)
    r_max = role_argmax(required, ff_array)

    total += r_min.size
    diverge += int((r_min != r_max).sum())
    attribue = r_min > 0
    if attribue.any():
        idx = np.flatnonzero(attribue)
        invalides += int((~cm[r_min[idx] - 1, idx]).sum())

print(f"fonctions                          : {len(dic_roles_skills)}")
print(f"  dont plusieurs niveaux           : {multi_role} "
      f"({100*multi_role/len(dic_roles_skills):.0f} %)")
print(f"recherches (fonction x pompier)    : {total}")
print(f"  réponses divergentes             : {diverge} ({100*diverge/total:.1f} %)")
print(f"  rôles attribués par argmin mais")
print(f"  incompatibles avec le pompier    : {invalides} ({100*invalides/total:.1f} %)")

La dernière ligne est le cœur du problème : sous `argmin`, ces affectations désignent un
pompier qui ne peut pas tenir le rôle. Le départ échoue alors au remplissage d'équipage
— c'est le compteur `rupture_ff`, mesuré à **17 694** avec le code de la thèse contre
**7 514** avec le code actuel, sur un flux d'événements identique.

Le chemin par lequel ça se propage :
`get_role_from_skills` → `get_roles_for_ff` → `gen_state`, donc l'état de **chaque**
décision, pour la baseline comme pour l'agent.

La baseline en souffre davantage : elle applique une règle fixe
(`min(potential_skills)`) et fait confiance aux niveaux qu'on lui donne. L'agent, lui,
apprend contre l'état qu'on lui présente — un biais systématique reste en partie
absorbable par l'apprentissage. D'où la remarque du commit : *« the baseline gains far
more than the agent, so the baseline/agent gap itself moves »*.